# TAPR: one-click GSM8K seed run

Trains **one** run (ORM baseline or TAPR) on GSM8K with Qwen2.5-3B + LoRA and saves everything to Google Drive.

**How to use**
1. Runtime → Change runtime type → **A100 GPU**.
2. In the next cell pick `MODE` and `SEED`, then **Runtime → Run all**.
3. Run the *other* mode in a **second notebook** (separate runtime). One run per runtime.

**If Colab disconnects:** reconnect, untick `RUN_SMOKE`, Run all again. The script resumes from the latest checkpoint (saved every 100 steps) in the same Drive folder.

Outputs: `MyDrive/tapr_runs/gsm_<mode>_s<seed>/` (checkpoints, adapter, training log) and `gsm_<mode>_s<seed>.log`.

In [ ]:
#@title 1. Configure the run
import os
MODE = "tapr"        #@param ["baseline", "tapr"]
SEED = 1             #@param {type:"integer"}
RUN_SMOKE = True     #@param {type:"boolean"}
LAMBDA = 0.3         #@param {type:"number"}

assert MODE in ("baseline", "tapr")
RUN_NAME = f"gsm_{MODE}_s{SEED}"
os.environ.update({
    "MODE": MODE, "SEED": str(SEED), "RUN_SMOKE": "1" if RUN_SMOKE else "0",
    "LAMBDA": str(LAMBDA), "RUN_NAME": RUN_NAME,
    "OUT_DIR": f"/content/drive/MyDrive/tapr_runs/{RUN_NAME}",
    "LOG": f"/content/drive/MyDrive/tapr_runs/{RUN_NAME}.log",
})
print(f"Run: {RUN_NAME} | smoke test first: {RUN_SMOKE} | lambda (tapr only): {LAMBDA}")

In [ ]:
#@title 2. Check the GPU
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> A100"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("torch", torch.__version__, "| CUDA", torch.version.cuda)

In [ ]:
#@title 3. Mount Google Drive (outputs live here so they survive disconnects)
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs("/content/drive/MyDrive/tapr_runs", exist_ok=True)

In [ ]:
%%bash
# 4. Get the code, install pinned packages, verify 4-bit works
set -eo pipefail
cd /content
if [ -d TAPR/.git ]; then git -C TAPR pull -q --ff-only; else git clone -q https://github.com/HarshRaj1607/TAPR TAPR; fi
cd TAPR
# Colab ships its own torch; install everything else from the pinned requirements.
grep -v '^torch' requirements.txt > /content/req_notorch.txt
pip install -q -r /content/req_notorch.txt
python - <<'PY'
import torch, trl, transformers, peft, bitsandbytes as bnb
print("trl", trl.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| bitsandbytes", bnb.__version__)
# The exact op that failed on CUDA-13 Colab with bitsandbytes < 0.48:
w = torch.randn(64, 64).half().cuda()
bnb.functional.quantize_4bit(w, quant_type="nf4")
print("bitsandbytes 4-bit quantization OK")
PY

In [ ]:
%%bash
# 5. Smoke test (5 steps, 20 problems; skipped if RUN_SMOKE is off)
set -eo pipefail
if [ "$RUN_SMOKE" = "1" ]; then
  cd /content/TAPR
  python hackathon/src/train_grpo.py --task gsm8k --mode "$MODE" --seed "$SEED" \
    --smoke_test --output_dir "/content/smoke_$RUN_NAME"
  echo "SMOKE TEST PASSED"
else
  echo "smoke test skipped"
fi

In [ ]:
%%bash
# 6. Full training run (resumes automatically from the latest checkpoint)
set -eo pipefail
cd /content/TAPR
EXTRA=""
if [ "$MODE" = "tapr" ]; then EXTRA="--lambda_val $LAMBDA"; fi
python hackathon/src/train_grpo.py --task gsm8k --mode "$MODE" --seed "$SEED" $EXTRA \
  --output_dir "$OUT_DIR" 2>&1 | tee -a "$LOG"
echo "RUN FINISHED: $RUN_NAME"

In [ ]:
#@title 7. Status: tail the log and list checkpoints (run any time, e.g. after reconnecting)
import os, glob
log, out = os.environ["LOG"], os.environ["OUT_DIR"]
print("checkpoints:", sorted(os.path.basename(p) for p in glob.glob(out + "/checkpoint-*")))
print("final adapter saved:", os.path.exists(out + "/run_summary.json") or os.path.isdir(out + "/adapter"))
print("--- last 2500 chars of log ---")
print(open(log).read()[-2500:] if os.path.exists(log) else "no log yet")